# controls

> Native form controls styled through their own states, raised to press and sunken to type into


In [ ]:
#| default_exp controls

Every form control we need is already a native HTML tag, and the browser already keeps track of its state. We style those tags directly, and we style their states through the pseudo-classes (`:disabled`, `:user-invalid`, …) and ARIA attributes (`aria-busy`, `aria-pressed`, …) that the browser provides, rather than tracking anything ourselves. Variants are bare attributes.

In [ ]:
from fastcore.utils import *
from fasthtml.common import Div, A, Span, Label, Input, Small, Form, Textarea, Select, Option, Legend, Menu, Li
from answerui.registry import reg, hypelt
from answerui.tokens import relief
from answerui.demo import example
from answerui.layout import Stack, Cluster
import answerui.base  # the reset this styling is written on

In [ ]:
from fasthtml.common import *
from answerui.demo import demo
from answerui.registry import reg
from fastcore.test import *
from fasthtml.svg import Path

## Button

We design our buttons with deliberate depth affordances to indicate 'clickability'. A raised face invites a press, and a sunken face shows that a button is already pressed. We keep the overall size of a button fixed while a press happens, moving only its label, bevels and shadows, so that a row of buttons stays level however many of them are down.

Some presses last longer than a click. While a request is in flight the button carries `aria-busy`, while a toggle is on it carries `aria-pressed`, and a button that opened a popover or a menu carries `aria-expanded` until it closes. In each case the button simply stays down, and the same affordance that invited the press now reports its state. Only the press itself moves the label; a button that is held stays sunken with its label centred, so a row of buttons stays aligned however many of them are down.

Busy also shows a spinner, but the button never grows or changes its wording to make room for it. The label fades, the spinner takes its place, and the label returns when the request ends. Busy is presentation only, so a control that must not be activated is `disabled`. A button that makes its own htmx request becomes busy by itself, because htmx marks it with `htmx-request` while the request runs. That busy face appears only once the request has lasted `--busy-delay`, 200ms, so a request that answers at once shows nothing, where `aria-busy` set by the server shows at once. A variant names its held look in `--face-held` and `--shadow-held`, which every held state and the delayed busy face read. A form marks itself rather than its button, so give the form `hx-indicator` naming the button.

A disabled button is flat and faded, printed on the surface rather than standing off it, because there is nothing to press. It keeps its own face, so a disabled primary still reads as the primary, and when it is enabled it rises out of the page.

The default button is the primary action and takes the accent colour. `secondary` puts it on the neutral surface with a hairline border instead. `quiet` has no face, border, or shadow at rest and inherits the colour around it, for actions that should not compete with the content; it shows a tint of that colour when hovered, and twice that tint when pressed or latched.

A link that acts as a button, carries the `button` attribute. It takes every treatment a button does, including the variants. It cannot be disabled or busy, since a link has no such states.

In [ ]:
_button_css = r"""
:is(button, a[button]) {
  --face: var(--v); --face-held: var(--v2); --shadow-held: var(--bevel-in), var(--sunken); --_shift: 0px; --busy-delay: 200ms;
  --pad-block: var(--space-2); --pad-inline: var(--space);
  display: inline-flex; align-items: center; justify-content: center; gap: var(--space-2);
  padding: calc(var(--pad-block) + var(--_shift)) calc(var(--pad-inline) - var(--_shift))
           calc(var(--pad-block) - var(--_shift)) calc(var(--pad-inline) + var(--_shift));
  font-weight: var(--weight1); line-height: var(--leading-1);
  background: var(--face); color: var(--n-3); text-decoration: none;
  border: 1px solid transparent; border-radius: var(--radius);
  box-shadow: var(--bevel), var(--raised);
  transition: box-shadow calc(var(--motion) / 2) var(--ease), padding calc(var(--motion) / 2) var(--ease), background var(--motion) var(--ease), opacity var(--motion) var(--ease), -webkit-text-fill-color var(--motion) var(--ease);
}
:is(button, a[button]):hover { --face: var(--v1); }
:is(button, a[button]):is(:active, [aria-busy=true], [aria-pressed=true], [aria-expanded=true]) { --face: var(--face-held); box-shadow: var(--shadow-held); }
:is(button, a[button]):active { --_shift: calc(1px * var(--relief)); transition-duration: calc(var(--motion) / 4); }
:is(button, a[button]):disabled { opacity: .5; --_shift: 0px; box-shadow: none; }
button:is([aria-busy=true], .htmx-request) { position: relative; }
button[aria-busy=true] { -webkit-text-fill-color: transparent; }
button > svg { transition: opacity var(--motion) var(--ease); }
button:is([aria-busy=true], .htmx-request) > svg { opacity: 0; }
button:is([aria-busy=true], .htmx-request)::before {
  content: ''; position: absolute; inset: 0; margin: auto; inline-size: 1.25em; block-size: 1.25em;
  border: 2px solid currentColor; border-inline-end-color: transparent; border-radius: var(--radius-full);
  animation: aui-spin calc(var(--motion) * 5) linear infinite;
}
:is(button, a[button]).htmx-request:not([aria-busy=true]) { animation: aui-sink 0s linear var(--busy-delay) both; }
button.htmx-request:not([aria-busy=true]) > svg { transition-delay: var(--busy-delay); }
button.htmx-request:not([aria-busy=true])::before { animation: aui-spin calc(var(--motion) * 5) linear infinite, aui-busy-in var(--motion) var(--ease) var(--busy-delay) both; }
@keyframes aui-sink { to { --face: var(--face-held); box-shadow: var(--shadow-held); -webkit-text-fill-color: transparent; } }
@keyframes aui-busy-in { from { opacity: 0; } }
"""

Button = hypelt('Button', sel='button, a[button]', layer='components', doc='Raised at rest, down while active, busy, or expanded, latched by `aria-pressed`; `secondary` for the lesser action; a link carrying `button` is styled the same way.', css=_button_css)

In [ ]:
Button.variant('secondary', r"""
--face: var(--n-2); color: var(--n2); border-color: var(--n-1);
--face-held: var(--n-2);
&:hover { --face: var(--v-3); }
&:is([aria-pressed=true], [aria-expanded=true]) { --face: var(--v-3); color: var(--v2); }
""", layer='components', doc='The lesser action: the neutral surface with a hairline border, and the accent wash when latched.')

Button.secondary [components] :is(button, a[button])[secondary]: The lesser action: the neutral surface with a hairline border, and the accent wash when latched.

In [ ]:
Button.variant('quiet', r'''
--face: transparent; color: inherit; border-color: transparent; box-shadow: none;
--face-held: oklch(from var(--tint) l c h / calc(alpha * 2)); --shadow-held: none;
&:hover { --face: var(--tint); }
&:is(:active, [aria-busy=true], [aria-pressed=true], [aria-expanded=true]) { --face: var(--face-held); }
&:active { --_shift: 0px; }
''', layer='components', doc='No face, border, or shadow at rest, for an action that should not compete: dismiss, close, and toolbar buttons.')

Button.quiet [components] :is(button, a[button])[quiet]: No face, border, or shadow at rest, for an action that should not compete: dismiss, close, and toolbar buttons.

Every face and state in a wrapping cluster: rest, secondary, a danger and a success context, disabled, busy, and latched:


In [ ]:
@example(Button)
def _button_example():
    return Cluster(Button('Save'), Button('Cancel', secondary=True), Button('Dismiss', quiet=True), A('Sign up', href='#', button=True), Button('Delete', danger=True), Button('Publish', success=True),
        Button('Locked', disabled=True), Button('Save', aria_busy='true'), Button('Pinned', aria_pressed='true'), Button('Pinned', secondary=True, aria_pressed='true'), Button('Pinned', quiet=True, aria_pressed='true'),
        gap='var(--space-2)')


In [ ]:
Button.demo()

```html
<div cluster style="--gap: var(--space-2);">
<button>Save</button><button secondary>Cancel</button><button quiet>Dismiss</button><a href="#" button>Sign up</a><button danger>Delete</button><button success>Publish</button><button disabled>Locked</button><button aria-busy="true">Save</button><button aria-pressed="true">Pinned</button><button secondary aria-pressed="true">Pinned</button><button quiet aria-pressed="true">Pinned</button></div>

```

Application styling can change the resting padding without rewriting the pressed state. Here `compact` changes the dimensions and `flat` removes relief. Declared as variants here, they belong to this example, not to the built-in button, and they appear in `doc(Button)` for as long as they are registered. To soften the whole application instead, set `:root { --relief: .5; }`.

In [ ]:
Button.variant('compact', '--pad-block: var(--space-3); --pad-inline: var(--space-2);', doc='Tighter padding.')
Button.variant('flat', '--relief: 0;', doc='No rims, shadows, or displacement.')

Button.flat [app] :is(button, a[button])[flat]: No rims, shadows, or displacement.

The flat buttons keep their colour states without rims, shadows, or displacement; the two pin buttons show unselected and selected states:

In [ ]:
demo(Cluster(
    Button('Compact', compact=True), Button('Flat', flat=True),
    Button('Pin', secondary=True, flat=True, aria_pressed='false'),
    Button('Pin', secondary=True, flat=True, aria_pressed='true'),
    gap='var(--space-2)'))

```html
<div cluster style="--gap: var(--space-2);">
<button compact>Compact</button><button flat>Flat</button><button secondary flat aria-pressed="false">Pin</button><button secondary flat aria-pressed="true">Pin</button></div>

```

A button that holds nothing but an icon should be square, so when a button's only child is an `svg` we make its inline padding match its block padding. Give such a button an `aria-label`, because the icon gives it no name:

In [ ]:
Button.rule('icon-only', '&:has(> svg:only-child) { --pad-inline: var(--pad-block); }', layer='components', doc='A button whose only child is an svg is square; give it an aria-label.')

Button.icon-only [components] button, a[button]: A button whose only child is an svg is square; give it an aria-label.

In [ ]:
cross = Svg(Path(d='M18 6 6 18'), Path(d='m6 6 12 12'), viewBox='0 0 24 24', fill='none', stroke='currentColor', stroke_width=2, stroke_linecap='round', stroke_linejoin='round', icon=True)
close = [Button(cross, aria_label='Close'), Button(cross, aria_label='Close', secondary=True), Button(cross, aria_label='Close', quiet=True)]
demo(Cluster(*close, Button(cross, 'Close', secondary=True), Button('Save'), gap='var(--space-2)'))

```html
<div cluster style="--gap: var(--space-2);">
<button aria-label="Close"><svg viewbox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round" icon><path d="M18 6 6 18"></path><path d="m6 6 12 12"></path></svg></button><button aria-label="Close" secondary><svg viewbox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round" icon><path d="M18 6 6 18"></path><path d="m6 6 12 12"></path></svg></button><button aria-label="Close" quiet><svg viewbox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round" icon><path d="M18 6 6 18"></path><path d="m6 6 12 12"></path></svg></button><button secondary><svg viewbox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round" icon><path d="M18 6 6 18"></path><path d="m6 6 12 12"></path></svg>Close</button><button>Save</button></div>

```

## Text controls

Where a button is raised, a control is sunken. It is a well cut into the page, the same lighting as a button inverted, so the eye reads it as somewhere to put something rather than something to press. A disabled well goes flat, as a disabled button does. These rules apply to the tags themselves, so a form written by hand looks the same as one built with `Field`. The well is for the text-like input types, textareas, and selects; a button made from `input`, and a file picker, keep their native look.

A number input is a text well without the spinner, since the up and down buttons are never how a number is entered. The keyboard's arrows still step the value.

Invalidity is a danger context. The browser sets `:user-invalid` once the user has interacted with a control, the server sets `aria-invalid`, and either one switches the control to the danger hue, so its border and error text take that colour with no rule of their own:

In [ ]:
reg.context(':user-invalid, [aria-invalid=true]', 'var(--hue-danger)')

_text_sel = 'input:is(:not([type]), [type=text], [type=email], [type=password], [type=search], [type=url], [type=tel], [type=number], [type=date], [type=time], [type=datetime-local], [type=month], [type=week])'
reg.rule('text-controls', rf"""
:is({_text_sel}, textarea, select) {{
  inline-size: 100%; padding: var(--space-2) var(--space-1);
  --face: var(--n-3); background: var(--face); color: var(--n2);
  border: 1px solid var(--n-1); border-radius: var(--radius); box-shadow: var(--bevel-in), var(--sunken);
  transition: border-color var(--motion) var(--ease), box-shadow var(--motion) var(--ease);
}}
:is(input, textarea, select):is(:user-invalid, [aria-invalid=true]) {{ border-color: var(--v); }}
:is({_text_sel}, textarea, select):disabled {{ --face: var(--n-2); color: var(--n); box-shadow: var(--bevel-in); }}
::placeholder {{ color: var(--n1); }}
input[type=number] {{ appearance: textfield; }}
input[type=number]::-webkit-inner-spin-button, input[type=number]::-webkit-outer-spin-button {{ appearance: none; margin: 0; }}
""", layer='components', doc='The well shared by text inputs, textareas, and selects, taking the danger hue when invalid.')

Input = hypelt('Input', sel=_text_sel, layer='components', doc='A text-like input, styled as a well by the text-controls rule.')
Textarea = hypelt('Textarea', layer='components', css='textarea { min-block-size: 5rem; resize: vertical; field-sizing: content; }',
    doc='A textarea, styled as a well by the text-controls rule; it grows with its content.')
Select = hypelt('Select', layer='components', doc='A select, styled as a well by the text-controls rule.')

Each text control on its own, at rest, and disabled:

In [ ]:
@example(Input)
def _input_example(): return Stack(Input(placeholder='Search', type='search', aria_label='Search'), Input(value='Locked', disabled=True, aria_label='Locked'), style='inline-size:16rem')

@example(Textarea)
def _textarea_example(): return Textarea('A note that grows with its content.', aria_label='Note', style='inline-size:16rem')

@example(Select)
def _select_example(): return Select(Option('Free'), Option('Team'), aria_label='Plan', style='inline-size:16rem')

In [ ]:
Input.demo()

```html
<div stack style="inline-size:16rem">
  <input placeholder="Search" type="search" aria-label="Search">
  <input value="Locked" disabled aria-label="Locked">
</div>

```

## Enabling a button

A Save button that is live before there is anything to save invites a click that does nothing. So a submit button can be rendered `disabled` and say, with `enable`, what brings it to life: `changed`, once any control in its form differs from the value it was rendered with, `valid`, once the form passes its own validation, or both. The browser already remembers every control's rendered value as its default, so nothing is copied, and a reset puts the button back to sleep along with the form. A form that htmx swaps in starts from its rendered state like any other.


In [ ]:
_enable_js = r"""
const f = el.form ?? el;
const run = () => {
    const changed = [...f.elements].some(c => c.matches('input:not([type=hidden], [type=submit], [type=button], [type=reset]), select, textarea')
        && (c.type === 'checkbox' || c.type === 'radio' ? c.checked !== c.defaultChecked
            : c.tagName === 'SELECT' ? [...c.options].some(o => o.selected !== o.defaultSelected) : c.value !== c.defaultValue));
    for (const b of f.querySelectorAll('button[enable]')) {
        const wants = b.getAttribute('enable').split(' ');
        b.disabled = !wants.every(w => w === 'changed' ? changed : w === 'valid' ? f.checkValidity() : true);
    }
};
e.type === 'reset' ? setTimeout(run) : run();"""

reg.rule('enable', on={('form:has(button[enable])', 'input'): _enable_js, ('form:has(button[enable])', 'change'): _enable_js, ('form:has(button[enable])', 'reset'): _enable_js, ('button[enable]', 'init'): _enable_js},
    layer='components', doc="A submit button rendered disabled with `enable='changed'`, `enable='valid'`, or both comes alive when its form has changed, or is valid, and goes back when it is not.")

enable [components]: A submit button rendered disabled with `enable='changed'`, `enable='valid'`, or both comes alive when its form has changed, or is valid, and goes back when it is not.

To see it work, change the name or type an email address. Save wakes only once the form is both changed and valid, and the email is required. Reset puts the form back, and Save goes back to sleep with it:


In [ ]:
demo(Form(Input(name='name', value='Ada', aria_label='Display name'), Input(name='email', type='email', required=True, placeholder='Email', aria_label='Email'),
    Cluster(Button('Save', type='submit', disabled=True, enable='changed valid'), Button('Reset', type='reset', secondary=True)), stack=True, style='inline-size: 24rem'))


```html
<form enctype="multipart/form-data" stack style="inline-size: 24rem">  <input name="name" value="Ada" aria-label="Display name">
  <input name="email" type="email" required placeholder="Email" aria-label="Email">
  <div cluster>
<button type="submit" disabled enable="changed valid">Save</button><button type="reset" secondary>Reset</button>  </div>
</form>
```

## Group

Controls that act as one share their edges: only the outer corners are rounded, and neighbours meet at a single hairline. A row of commands is a `Toolbar`, HTML's `menu` with each command in an item, such as a pager or a set of toggles. Which toggle is down is state, and the server sets it with `aria-pressed`; the toolbar is only layout. A text input and its one action are a `Group`: a search box and its button, or a name and Create. The input takes the width the button leaves, and while the text has focus the ring traces the whole group, because the pair is one control.

`inset` puts the button inside the input's well instead, which suits a small action that belongs to the text, such as copying or clearing it. Either way the button starts disabled and `enable` wakes it once the text calls for it, so at rest the group reads as a field rather than a pair of controls.


In [ ]:
_group_css = r"""
[group] { display: inline-flex; align-items: stretch; margin: 0; padding: 0; list-style: none; border-radius: var(--radius); --_radius: var(--radius); --_small: var(--radius-1); }
[group] > li { display: flex; }
[group] > *, [group] > li > * { --radius: 0; }
[group] > :first-child, [group] > li:first-child > * { border-start-start-radius: var(--_radius); border-end-start-radius: var(--_radius); }
[group] > :last-child, [group] > li:last-child > * { border-start-end-radius: var(--_radius); border-end-end-radius: var(--_radius); }
[group] > * + * { margin-inline-start: -1px; }
[group]:has(> :is(input, select)) { inline-size: 100%; }
[group] > :is(input, select) { flex: 1; min-inline-size: 0; }
[group] > :is(button, a[button]) { flex: none; }
[group]:has(> :is(input, select):focus-visible) { outline: var(--ring); outline-offset: 2px; }
[group] > :is(input, select):focus-visible { outline: none; }
"""

@hypelt(sel='[group]', layer='components', css=_group_css)
def Group(*c, **attrs):
    "A text input and its action sharing an edge; `inset` puts the action inside the well"
    return Div(*c, group=True, **attrs)

@hypelt(sel='menu[group]', layer='components')
def Toolbar(*controls, **attrs):
    "A row of commands sharing their edges: a `menu` with each control as an item"
    return Menu(*[Li(c) for c in controls], group=True, **attrs)

Group.variant('inset', r"""
--face: var(--n-3); background: var(--face); border: 1px solid var(--n-1); box-shadow: var(--bevel-in), var(--sunken);
align-items: center; padding-inline-end: var(--space-3);
& > * { margin: 0; --radius: var(--_small); }
& > :first-child, & > :last-child { border-radius: var(--_small); }
& > :is(input, select) { border: 0; background: transparent; box-shadow: none; }
& > :is(button, a[button]), & > [tooltip] > :is(button, a[button]) { --pad-block: var(--space-3); --pad-inline: var(--space-2); }
""", layer='components', doc='The action inside the well; the action starts disabled and `enable` wakes it.')

Group.inset [components] :is([group])[inset]: The action inside the well; the action starts disabled and `enable` wakes it.

Two groups: a name with its Create button, and a name being edited in an inset well. Type into the first and Create wakes once there is a name. Change the second and its Save wakes:


In [ ]:
@example(Group)
def _group_example():
    return Stack(Form(Group(Input(name='name', placeholder='Instance name', required=True, aria_label='Instance name'), Button('Create', type='submit', disabled=True, enable='valid'))),
        Form(Group(Input(name='name', value='personal', aria_label='Name'), Button('Save', type='submit', disabled=True, enable='changed valid'), inset=True)),
        style='inline-size: 24rem')

In [ ]:
Group.demo()

```html
<div stack style="inline-size: 24rem">
<form enctype="multipart/form-data">    <div group>
      <input name="name" placeholder="Instance name" required aria-label="Instance name">
<button type="submit" disabled enable="valid">Create</button>    </div>
</form><form enctype="multipart/form-data">    <div group inset>
      <input name="name" value="personal" aria-label="Name">
<button type="submit" disabled enable="changed valid">Save</button>    </div>
</form></div>

```

Two toolbars: a pager, and a toggle bar with one button latched and one disabled. The bar is a named group, since its buttons are toggles rather than a form choice; an exclusive choice is what native radios are for. A toggle's state belongs to the server, so to demonstrate it here a demo-only rule flips `aria-pressed` on click:


In [ ]:
reg.rule('demo-toolbar-toggle', sel='menu[group]', on={('& button[aria-pressed]', 'click'): r"el.setAttribute('aria-pressed', String(el.getAttribute('aria-pressed') !== 'true'))"},
    doc='Demo only: flips aria-pressed in place of the server.')

demo-toolbar-toggle [app] menu[group]: Demo only: flips aria-pressed in place of the server.

In [ ]:
@example(Toolbar)
def _toolbar_example():
    return Cluster(Toolbar(Button('Previous', secondary=True, type='button'), Button('Next', secondary=True, type='button'), aria_label='Pages'),
        Toolbar(Button('Bold', secondary=True, aria_pressed='true'), Button('Italic', secondary=True, aria_pressed='false'),
            Button('Underline', secondary=True, aria_pressed='false', disabled=True), aria_label='Text style'))

In [ ]:
Toolbar.demo()

```html
<div cluster>
<menu group aria-label="Pages">    <li>
<button secondary type="button">Previous</button>    </li>
    <li>
<button secondary type="button">Next</button>    </li>
</menu><menu group aria-label="Text style">    <li>
<button secondary aria-pressed="true">Bold</button>    </li>
    <li>
<button secondary aria-pressed="false">Italic</button>    </li>
    <li>
<button secondary aria-pressed="false" disabled>Underline</button>    </li>
</menu></div>

```

## Click to edit

A value that is read far more often than it is changed should not sit in a well, which invites typing, nor in plain text, which gives no sign it can change. `editable` draws it as the dashed ghost of the well it becomes: the same box, padding and corners, with a dashed hairline in place of the sunken face. No field exists until the click, and nothing depends on hover. The click fetches the form, and the app swaps it in where the ghost was: a single well for a name, or a whole form for a value made of parts. Inside a form give it `type="button"`. Its text is the value, so name the action with `aria-label`:


In [ ]:
Button.variant('editable', r'''
--face: transparent; --pad-block: var(--space-2); --pad-inline: var(--space-1);
--face-held: oklch(from var(--tint) l c h / calc(alpha * 2)); --shadow-held: none;
inline-size: 100%; justify-content: start; text-align: start;
color: inherit; font: inherit; cursor: text;
border: 1px dashed var(--n); box-shadow: none;
&:hover { --face: var(--tint); }
&:is(:active, [aria-busy=true], [aria-pressed=true], [aria-expanded=true]) { --face: var(--face-held); }
&:active { --_shift: 0px; }
''', layer='components', doc='A value drawn as the dashed ghost of the well it becomes; click to edit, and the app swaps the form in.')

Button.editable [components] :is(button, a[button])[editable]: A value drawn as the dashed ghost of the well it becomes; click to edit, and the app swaps the form in.

To demonstrate this without a server, a demo-only rule swaps each value for its form and back. The first value is a name: click it and a well opens in its place, with a Save button that wakes once the text changes. The second is a schedule: click it and a small form opens instead, a day and a time. Either way, Save puts the new value back where the ghost was, and a click anywhere else, or Escape, puts the old one back:


In [ ]:
reg.rule('demo-editable', sel='[editable-demo]', on={
    ('& button[editable]', 'click'): r"const p = el.closest('[editable-demo]'); p.firstElementChild.hidden = true; p.lastElementChild.hidden = false; p.querySelector('input, select').focus()",
    ('& form', 'submit'): r"e.preventDefault(); const p = el.closest('[editable-demo]'); p.querySelector('button[editable]').textContent = el.dataset.text.replace(/\{(\w+)\}/g, (_, n) => el.elements[n].value); el.hidden = true; p.firstElementChild.hidden = false",
    ('& form', 'keydown'): r"if (e.key === 'Escape') { el.reset(); el.hidden = true; el.previousElementSibling.hidden = false }",
    (':root', 'pointerdown'): r"for (const f of document.querySelectorAll('[editable-demo] > form:not([hidden])')) if (!f.contains(e.target)) { f.reset(); f.hidden = true; f.previousElementSibling.hidden = false }",
}, doc='Demo only: swaps the value for its form and back in place of the server; a click outside or Escape cancels.')

def _editable_demo(value, form, text, **attrs):
    "The value at rest and the form it opens into; `text` says how the form's fields make the value"
    return Div(Button(value, editable=True, type='button', **attrs), Form(form, data_text=text, hidden=True), editable_demo=True)


In [ ]:
days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']
demo(Stack(
    _editable_demo('Untitled-1', Group(Input(name='name', value='Untitled-1', aria_label='Name'), Button('Save', type='submit', disabled=True, enable='changed valid'), inset=True),
        '{name}', aria_label='Rename Untitled-1'),
    _editable_demo('Every Monday at 09:00', Group(Select(*[Option(d, selected=d == 'Monday') for d in days], name='day', aria_label='Day'),
        Input(type='time', name='time', value='09:00', aria_label='Time'), Button('Save', type='submit', disabled=True, enable='changed valid')),
        'Every {day} at {time}', aria_label='Change the schedule'),
    style='inline-size: 24rem'))


```html
<div stack style="inline-size: 24rem">
  <div editable-demo>
<button editable type="button" aria-label="Rename Untitled-1">Untitled-1</button><form enctype="multipart/form-data" data-text="{name}" hidden>      <div group inset>
        <input name="name" value="Untitled-1" aria-label="Name">
<button type="submit" disabled enable="changed valid">Save</button>      </div>
</form>  </div>
  <div editable-demo>
<button editable type="button" aria-label="Change the schedule">Every Monday at 09:00</button><form enctype="multipart/form-data" data-text="Every {day} at {time}" hidden>      <div group>
<select name="day" aria-label="Day"><option selected>Monday</option><option>Tuesday</option><option>Wednesday</option><option>Thursday</option><option>Friday</option></select>        <input type="time" name="time" value="09:00" aria-label="Time">
<button type="submit" disabled enable="changed valid">Save</button>      </div>
</form>  </div>
</div>

```

## Switch

Typically, a switch turns something on or off, doing so straight away, where a checkbox records a choice for a form to submit. The two are the same control to the browser: a switch is a checkbox with `role="switch"`, and the browser supplies the state, the keyboard handling, and the announcement. `Switch` only pairs the type with the role. Its look is built from the two treatments we already have: the track is a well, and the thumb is raised. Turning it on colours the track and slides the thumb across. In a `Field` it sits before its label, as a checkbox does:

In [ ]:
_switch_css = r"""
input[role=switch] { appearance: none; --face: var(--n-3); inline-size: 2.75em; block-size: 1.5em; margin: 0; padding: 2px;
  background: var(--face); border: 1px solid var(--n-1); border-radius: var(--radius-full); box-shadow: var(--bevel-in), var(--sunken);
  display: inline-grid; align-items: center;
  transition: background var(--motion) var(--ease), box-shadow var(--motion) var(--ease); }
input[role=switch]::before { content: ''; --face: var(--n-2); inline-size: calc(1.5em - 6px); block-size: calc(1.5em - 6px); border-radius: var(--radius-full);
  background: var(--face); box-shadow: var(--bevel), var(--raised);
  transition: translate var(--motion) var(--ease), background var(--motion) var(--ease); }
input[role=switch]:checked { --face: var(--v); }
input[role=switch]:checked::before { translate: 1.25em 0; --face: var(--n-3); }
input[role=switch]:disabled { --face: var(--n-2); box-shadow: var(--bevel-in); }
input[role=switch]:disabled::before { --face: var(--n-1); box-shadow: var(--bevel); }
"""

@hypelt(sel='input[role=switch]', layer='components', css=_switch_css)
def Switch(**attrs):
    "A checkbox with the switch role"
    return Input(type='checkbox', role='switch', **attrs)

Off, on, and disabled:

In [ ]:
@example(Switch)
def _switch_example():
    return Cluster(Switch(aria_label='Off'), Switch(checked=True, aria_label='On'), Switch(disabled=True, aria_label='Disabled'))

In [ ]:
Switch.demo()

```html
<div cluster>
  <input type="checkbox" role="switch" aria-label="Off">
  <input type="checkbox" role="switch" checked aria-label="On">
  <input type="checkbox" role="switch" disabled aria-label="Disabled">
</div>

```

## Checkbox and radio

The browser's own checkbox and radio follow the hue, but they have none of our depth, so we set `appearance: none` and restyle them ourselves. Each is a small well. At this size we drop the rims and keep only the sunken shade. `Checkbox` and `Radio` only set the type:

In [ ]:
_choice_css = r"""
input[type=checkbox]:not([role=switch]), input[type=radio] { appearance: none; --face: var(--n-3); inline-size: 1.15em; block-size: 1.15em; margin: 0;
  background: var(--face); border: 1px solid var(--n-1); border-radius: var(--radius-1); box-shadow: var(--sunken);
  display: inline-grid; place-content: center;
  transition: background var(--motion) var(--ease), border-color var(--motion) var(--ease); }
input[type=checkbox]:not([role=switch])::before, input[type=radio]::before { content: ''; inline-size: .6em; block-size: .6em; background: var(--n-3); scale: 0; transition: scale var(--motion) var(--ease); }
:is(input[type=checkbox]:not([role=switch]), input[type=radio]):disabled { --face: var(--n-2); border-color: var(--n-1); }
"""
reg.rule('choice-controls', _choice_css, layer='components', doc='The small well and the mark shared by checkboxes and radios.')

_checkbox_css = r"""
input[type=checkbox]:not([role=switch])::before { clip-path: polygon(14% 44%, 0 65%, 50% 100%, 100% 16%, 80% 0%, 43% 62%); }
input[type=checkbox]:not([role=switch]):is(:checked, :indeterminate) { --face: var(--v); border-color: var(--v); }
input[type=checkbox]:not([role=switch]):is(:checked, :indeterminate)::before { scale: 1; }
input[type=checkbox]:not([role=switch]):indeterminate::before { clip-path: none; block-size: .15em; }
input[type=checkbox]:not([role=switch]):is(:checked, :indeterminate):disabled { --face: var(--n-1); border-color: var(--n-1); }
"""

@hypelt(sel='input[type=checkbox]:not([role=switch])', layer='components', css=_checkbox_css)
def Checkbox(**attrs):
    "A checkbox; the choice-controls rule draws its well"
    return Input(type='checkbox', **attrs)


In [ ]:
_radio_css = r"""
input[type=radio] { border-radius: var(--radius-full); }
input[type=radio]::before { border-radius: var(--radius-full); background: var(--v); }
input[type=radio]:checked { border-color: var(--v); }
input[type=radio]:checked::before { scale: 1; }
input[type=radio]:checked:disabled::before { background: var(--n); }
"""

@hypelt(sel='input[type=radio]', layer='components', css=_radio_css)
def Radio(**attrs):
    "A radio button; the choice-controls rule draws its well"
    return Input(type='radio', **attrs)

Unchecked, checked, disabled, and checked and disabled:

In [ ]:
@example(Checkbox)
def _checkbox_example():
    return Cluster(Checkbox(aria_label='Unchecked'), Checkbox(checked=True, aria_label='Checked'),
        Checkbox(disabled=True, aria_label='Disabled'), Checkbox(checked=True, disabled=True, aria_label='Checked and disabled'))


In [ ]:
Checkbox.demo()

```html
<div cluster>
  <input type="checkbox" aria-label="Unchecked">
  <input type="checkbox" checked aria-label="Checked">
  <input type="checkbox" disabled aria-label="Disabled">
  <input type="checkbox" checked disabled aria-label="Checked and disabled">
</div>

```

Radios in a group, unchecked, checked, and checked and disabled:

In [ ]:
@example(Radio)
def _radio_example():
    return Cluster(Radio(name='r', aria_label='Unchecked'), Radio(name='r', checked=True, aria_label='Checked'), Radio(name='rd', checked=True, disabled=True, aria_label='Checked and disabled'))

In [ ]:
Radio.demo()

```html
<div cluster>
  <input type="radio" name="r" aria-label="Unchecked">
  <input type="radio" name="r" checked aria-label="Checked">
  <input type="radio" name="rd" checked disabled aria-label="Checked and disabled">
</div>

```

## Range

A slider is the browser's `input[type=range]`, restyled as a well with a raised thumb. The one thing we add is a fill along the track up to the thumb. CSS cannot know where the thumb is, so a small behavior reads the slider's value whenever it changes and writes the fill, computed from the control's own bounds, into `--p`. A value set from a script does not fire that event, so such a change dispatches `input` to update the fill.

The track and the thumb are not elements of their own. They are boxes inside the input that the browser exposes as pseudo-elements, and each engine names them differently: WebKit, which Chrome and Safari share, has `::-webkit-slider-runnable-track` and `::-webkit-slider-thumb`, and Firefox has `::-moz-range-track` and `::-moz-range-thumb`. A browser drops a whole rule whose selector list names a pseudo-element it does not know, so the two spellings cannot share a rule. `_range_parts` writes the rules once and the stylesheet carries them under both names. Each part has a face of its own, the track the page colour and the thumb the accent, so each derives its own rims with `relief`. Two differences between the engines stay outside the shared rules: WebKit leaves the thumb at the top of the track where Firefox centres it, and Firefox gives the thumb a border of its own.

In [ ]:
_fill = r"const lo = el.min === '' ? 0 : +el.min, hi = el.max === '' ? 100 : +el.max; el.style.setProperty('--p', (hi > lo ? (el.value - lo) / (hi - lo) * 100 : 0) + '%')"
_range_on = {
    ('&', 'init'): _fill,
    ('&', 'input'): _fill,
    ('form:has(input[type=range])', 'reset'): r"""
await new Promise(r => setTimeout(r, 0));
for (const range of [...el.elements].filter(i => i.matches('input[type=range]'))) { const el = range; """ + _fill + " }"}

def _range_parts(track, thumb):
    "The track and thumb rules for one engine's pseudo-elements"
    return relief(f'input[type=range]::{track}, input[type=range]::{thumb}') + rf"""
input[type=range]::{track} {{ --face: var(--n-3); block-size: var(--track); border-radius: var(--radius-full); border: 1px solid var(--n-1);
  background: linear-gradient(to right, var(--v) var(--p), var(--face) 0); box-shadow: var(--bevel-in), var(--sunken); }}
input[type=range]::{thumb} {{ appearance: none; --face: var(--v); inline-size: var(--thumb); block-size: var(--thumb); border-radius: var(--radius-full);
  background: var(--face); box-shadow: var(--bevel), var(--raised); transition: box-shadow calc(var(--motion) / 2) var(--ease); }}
input[type=range]:active::{thumb} {{ box-shadow: var(--bevel-in), var(--sunken); }}
input[type=range]:disabled::{track} {{ background: linear-gradient(to right, var(--n) var(--p), var(--n-2) 0); }}
input[type=range]:disabled::{thumb} {{ --face: var(--n-1); box-shadow: var(--bevel); }}
"""

_range_css = r"""
input[type=range] { appearance: none; inline-size: 100%; block-size: 1.5em; margin: 0; background: none;
  --track: .6em; --thumb: 1.25em; --p: 0%; }
input[type=range]::-webkit-slider-thumb { margin-block-start: calc((var(--track) - var(--thumb)) / 2 - 1px); }
input[type=range]::-moz-range-thumb { border: 0; }
""" + _range_parts('-webkit-slider-runnable-track', '-webkit-slider-thumb') + _range_parts('-moz-range-track', '-moz-range-thumb')

@hypelt(sel='input[type=range]', layer='components', on=_range_on, css=_range_css)
def Range(**attrs):
    "A slider"
    return Input(type='range', **attrs)

A slider at 35, a disabled one at 65, one at 8 of 10, and one at -5 in a range that ends at zero, where a missing bound and a bound of zero must not be confused.

In [ ]:
@example(Range)
def _range_example():
    return Stack(Range(value=35, aria_label='Brightness'), Range(value=65, disabled=True, aria_label='Locked'), Range(min=0, max=10, value=8, aria_label='Rating'),
        Range(min=-10, max=0, value=-5, aria_label='Offset'),
        style='inline-size:16rem')

In [ ]:
Range.demo()

```html
<div stack style="inline-size:16rem">
  <input type="range" value="35" aria-label="Brightness">
  <input type="range" value="65" disabled aria-label="Locked">
  <input type="range" min="0" max="10" value="8" aria-label="Rating">
  <input type="range" min="-10" max="0" value="-5" aria-label="Offset">
</div>

```

## Field

A control rarely stands alone. It needs a label, often a line of help, and sometimes an error, and each of these has to be wired to the control so that assistive technology reads them together. `Field` does that wiring. It wraps the control in its label, which associates the two without an id, and puts a checkbox or radio before its label text rather than after. That label is only as wide as the box and its text, so a click in the empty space beside it does nothing. A `hint` and an `error` sit beneath, linked from the control's `aria-describedby`, and an error also marks the control `aria-invalid`. What counts as invalid is the application's decision; `Field` only shows the message it is given.


In [ ]:
_field_css = r"""
[field], [field] > label { display: grid; gap: var(--space-3); font-size: var(--text-1); }
[field] > label { font-weight: var(--weight1); }
[field] > small { font-weight: var(--weight); color: var(--n1); }
small[danger] { color: var(--v2); }
[field] > label:has(> :is([type=checkbox], [type=radio])) { grid-template-columns: auto 1fr; justify-self: start; align-items: center; gap: var(--space-2); font-weight: var(--weight); }
[field]:has(:disabled) > label { color: var(--n); }
"""

@hypelt(sel='[field]', layer='components', css=_field_css)
def Field(
    label,      # Label text
    control,    # The control
    hint=None,  # Help text shown under the control
    error=None, # Error message; marks the control invalid
    **attrs,    # Attributes for the field
):
    "A labelled control with optional help and error text"
    boxed = control.attrs.get('type') in ('checkbox', 'radio')
    descriptions = []
    if hint: descriptions.append(Small(hint, id=True))
    if error:
        descriptions.append(Small(error, id=True, danger=True))
        control(aria_invalid='true')
    if descriptions:
        ids = [control.attrs.get('aria-describedby', ''), *(o.id for o in descriptions)]
        control(aria_describedby=' '.join(filter(None, ids)))
    return Div(Label(*([control, label] if boxed else [label, control])), *descriptions, field=True, **attrs)


A submit button that waits on `enable='valid'` never submits an invalid form, so the browser's validation bubble, which only appears on submit, never shows. The field's `validity` rule shows that message instead: once the user has left a control invalid, the browser's own text for it appears under the field as an error, and goes when the value is valid again. An error the server rendered is untouched.

In [ ]:
Field.rule('validity', on={
    ('& :is(input, select, textarea)', 'focusout'): r"""
const f = el.closest('[field]'); let m = f.querySelector(':scope > small[validation]');
if (!el.matches(':user-invalid')) { m?.remove(); return }
if (!m) { m = document.createElement('small'); m.setAttribute('validation', ''); m.setAttribute('danger', ''); f.append(m) }
m.textContent = el.validationMessage""",
    ('& :is(input, select, textarea)', 'input'): "if (el.validity.valid) el.closest('[field]').querySelector(':scope > small[validation]')?.remove()",
}, layer='components', doc="Once the user leaves a control invalid, the browser's own validation message appears under the field; it goes when the value is valid again.")

Field.validity [components] [field]: Once the user leaves a control invalid, the browser's own validation message appears under the field; it goes when the value is valid again.

A server answers an invalid submission with the form again and its message. Mark that message `validation`, as a `small` outside any field, and it clears as soon as the user edits a control, since it described the values that were sent. A message inside a field belongs to that field's control and follows the rule above.

In [ ]:
reg.rule('form-validation', sel='form', on={('& :is(input, select, textarea)', 'input'): r"for (const m of el.form.querySelectorAll('small[validation]')) if (!m.closest('[field]')) m.remove()"},
    layer='components', doc='A validation message a form came back with, a `small` marked `validation` outside any field, clears once the user edits a control, since it described the values that were sent.')

form-validation [components] form: A validation message a form came back with, a `small` marked `validation` outside any field, clears once the user edits a control, since it described the values that were sent.

Every kind of control in one form. The username keeps its hint alongside a server-supplied error. Type an invalid address into the email field and leave it to see the browser's validation treatment:

In [ ]:
@example(Field)
def _field_example():
    return Form(
        Field('Name', Input(name='name', placeholder='Ada Lovelace')),
        Field('Email', Input(name='email', type='email', required=True), hint='We never share it.'),
        Field('Notes', Textarea(name='notes', placeholder='Anything else?')),
        Field('Plan', Select(Option('Free'), Option('Team'), name='plan')),
        Field('Send me updates', Checkbox(name='updates', checked=True)),
        Field('Email me about mentions', Switch(name='mentions', checked=True)),
        Field('Weekly digest', Radio(name='digest', checked=True)),
        Field('Volume', Range(name='volume')),
        Field('Region', Input(name='region', value='Locked', disabled=True)),
        Field('Username', Input(name='username', value='taken'), hint='Shown on your profile.', error='That name is in use.'),
        Cluster(Button('Save'), Button('Cancel', secondary=True, type='button'), gap='var(--space-2)'),
        stack=True)


In [ ]:
Field.demo()

```html
<form enctype="multipart/form-data" stack>  <div field>
<label>Name      <input name="name" placeholder="Ada Lovelace">
</label>  </div>
  <div field>
<label>Email      <input name="email" type="email" required aria-describedby="_vZO5YlsmRpy6ESrwQRBeYQ">
</label><small id="_vZO5YlsmRpy6ESrwQRBeYQ">We never share it.</small>  </div>
  <div field>
<label>Notes<textarea name="notes" placeholder="Anything else?"></textarea></label>  </div>
  <div field>
<label>Plan<select name="plan"><option>Free</option><option>Team</option></select></label>  </div>
  <div field>
<label>      <input type="checkbox" name="updates" checked>
Send me updates</label>  </div>
  <div field>
<label>      <input type="checkbox" role="switch" name="mentions" checked>
Email me about mentions</label>  </div>
  <div field>
<label>      <input type="radio" name="digest" checked>
Weekly digest</label>  </div>
  <div field>
<label>Volume      <input type="range" name="volume">
</label>  </div>
  <div field>
<label>Region      <input name="region" value="Locked" disabled>
</label>  </div>
  <div field>
<label>Username      <input name="username" value="taken" aria-invalid="true" aria-describedby="_Rrb8s6L4RZOkTVpLuHfKJw _UJ2sMMyCRK2n5p0-xLxrvg">
</label><small id="_Rrb8s6L4RZOkTVpLuHfKJw">Shown on your profile.</small><small danger id="_UJ2sMMyCRK2n5p0-xLxrvg">That name is in use.</small>  </div>
  <div cluster style="--gap: var(--space-2);">
<button>Save</button><button secondary type="button">Cancel</button>  </div>
</form>
```

Attributes passed to `Field` go on the field itself, so a variant declared on `Field` applies where its constructor puts the attribute. The control keeps its own attributes, and an existing description stays linked alongside the hint and error:

In [ ]:
notes = Textarea(name='notes', aria_describedby='notes-policy')
field = Field('Notes', notes, hint='Keep it brief.', error='Please remove the account number.', compact=True)
assert notes.attrs['aria-invalid'] == 'true'
assert notes.attrs['aria-describedby'].split() == ['notes-policy', *(o.id for o in field.children[1:])]
test_eq(field.attrs['compact'], True)
notes.attrs


{'name': 'notes',
 'aria-describedby': 'notes-policy _h6pj69MHRte64_HY9olwxg _-7kccwWETWm_w59m1yMMQA',
 'aria-invalid': 'true'}

## Fieldset

A fieldset groups fields under a legend. It draws a hairline panel and owns its padding; spacing between its fields is the layout's job, as everywhere else:

In [ ]:
_fieldset_css = r"""
fieldset { margin: 0; min-inline-size: 0; border: 1px solid var(--n-1); border-radius: var(--radius); padding: var(--space); }
legend { font-weight: var(--weight1); padding-inline: var(--space-2); }
"""

Fieldset = hypelt('Fieldset', layer='components', doc='A bordered group of fields under a legend.', css=_fieldset_css)

Two checkbox fields under a legend:

In [ ]:
@example(Fieldset)
def _fieldset_example():
    return Fieldset(Legend('Notifications'),
        Field('Email me about mentions', Input(type='checkbox', checked=True)),
        Field('Email me a weekly digest', Input(type='checkbox')),
        stack=True, style='--gap: var(--space-2)')

In [ ]:
Fieldset.demo()

```html
<fieldset stack style="--gap: var(--space-2)"><legend>Notifications</legend>  <div field>
<label>      <input type="checkbox" checked>
Email me about mentions</label>  </div>
  <div field>
<label>      <input type="checkbox">
Email me a weekly digest</label>  </div>
</fieldset>
```

## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()